# Sprint 4 — Webinar 13 (Teórico) — Versión Mejorada## Journey → Funnel → Cohorts (SQL con DuckDB en Jupyter)**Programa:** Data Analytics  **Sprint:** 4  **Duración:** 100 minutos  **Modalidad:** Teórica con ejercicios prácticos ejecutables  **Motor SQL:** DuckDB (in-process, sin servidor externo)> En esta sesión aprenderás a reconstruir el recorrido de un usuario (**Data Journey**), a escribir consultas modulares con **CTEs**, y a construir **funnels de conversión** y **tablas de retención por cohorts** — todo con SQL ejecutable directamente en este notebook.

<div style="text-align: center">    <img src="https://raw.githubusercontent.com/ljpiere/tpdata_python/main/images/w1s1_2.png" width="400"></div>

## Agenda sugerida (100 minutos)| Tiempo | Bloque | Contenido ||-------:|--------|-----------|| 0 – 5 | Setup | Instalar DuckDB, crear tablas y cargar datos || 5 – 25 | Sección 1 | **Data Journey:** qué es, por qué importa, etapas, campos clave, validación de datos || 25 – 45 | Sección 2 | **CTEs (Common Table Expressions):** sintaxis, ejemplos progresivos, buenas prácticas || 45 – 65 | Sección 3 | **Funnel Analysis:** construir funnels con CTEs, drop-off, conversión || 65 – 85 | Sección 4 | **Cohort Retention:** agrupar por cohorte, tabla de retención, segmentación || 85 – 95 | Sección 5 | **Ejercicios integradores** (dificultad creciente) || 95 – 100 | Cierre | Reflexión, Q&A, próximos pasos |

## Objetivos de AprendizajeAl finalizar la sesión, el/la estudiante podrá:1. **Definir** qué es un *Data Journey* / *User Journey* y explicar su importancia para el negocio.2. **Identificar** las etapas típicas de un journey y los campos necesarios en la base de datos (`user_id`, `event_ts`, `event_name`, `props`).3. **Validar** la calidad de los datos antes de cualquier análisis (duplicados, faltantes, timestamps).4. **Escribir CTEs** (`WITH … AS`) de complejidad creciente para modularizar consultas SQL.5. **Construir un funnel de conversión** con SQL y calcular drop-off y tasas de conversión.6. **Generar tablas de retención** por cohort (mensual/semanal) y segmentar por atributos.7. **Interpretar** resultados para proponer hipótesis de negocio accionables.

---# 0) Setup: Entorno SQL con DuckDB en JupyterDuckDB es un motor analítico SQL que corre **dentro del mismo proceso de Python** — no necesitas instalar ni configurar un servidor externo. Es ideal para aprender SQL y para prototipado rápido.> **¿Por qué DuckDB?** Es rápido, soporta SQL estándar, se integra con pandas y funciona directamente en un notebook.

In [ ]:
# Paso 1: Instalar DuckDB (solo la primera vez)!pip install duckdb -q

In [ ]:
# Paso 2: Importar y crear conexión en memoriaimport duckdb# Conexión en memoria (los datos viven mientras el notebook esté abierto)con = duckdb.connect(database=':memory:')# Helper para ejecutar SQL y mostrar resultados como tabladef sql(query):    """Ejecuta una consulta SQL y muestra el resultado como DataFrame."""    return con.execute(query).fetchdf()def sql_exec(query):    """Ejecuta una sentencia SQL sin retornar resultados (DDL, INSERT, etc.)."""    con.execute(query)    print("✅ DuckDB listo. Versión:", duckdb.__version__)

### 0.1 Crear tablas y cargar datosVamos a crear dos tablas que simulan un e-commerce:- **`users`** — Usuarios con fecha de registro, plan, canal de adquisición y dispositivo.- **`events`** — Eventos de comportamiento (session_start, view_item, add_to_cart, begin_checkout, purchase).> **Nota:** Estos mismos scripts funcionan en SQL Workbench o SQLite con mínimos ajustes.

In [ ]:
# ============================================================# Crear tablas (DDL)# ============================================================sql_exec("""-- Usuarios con atributos para segmentaciónCREATE TABLE IF NOT EXISTS users (  user_id     INTEGER PRIMARY KEY,  signup_ts   TIMESTAMP NOT NULL,  plan        VARCHAR NOT NULL,       -- 'free' o 'paid'  channel     VARCHAR NOT NULL,       -- 'organic','ads','referral','email'  device      VARCHAR NOT NULL        -- 'web','android','ios');""")sql_exec("""-- Tabla de eventos tipo GA4 minimalistaCREATE TABLE IF NOT EXISTS events (  event_id    INTEGER PRIMARY KEY,  user_id     INTEGER REFERENCES users(user_id),  event_name  VARCHAR NOT NULL,  event_ts    TIMESTAMP NOT NULL,  props       VARCHAR NOT NULL DEFAULT '{}');""")print("✅ Tablas 'users' y 'events' creadas correctamente.")

In [ ]:
# ============================================================# Insertar datos de ejemplo (Seed Data)# ============================================================# --- Usuarios (cohortes de enero y febrero 2025) ---sql_exec("""INSERT INTO users (user_id, signup_ts, plan, channel, device) VALUES  (1,  '2025-01-02 09:10:00', 'free',  'organic',  'web'),  (2,  '2025-01-03 14:20:00', 'paid',  'ads',      'ios'),  (3,  '2025-01-05 08:05:00', 'free',  'referral', 'android'),  (4,  '2025-01-10 18:42:00', 'free',  'organic',  'web'),  (5,  '2025-01-12 11:11:00', 'paid',  'email',    'web'),  (6,  '2025-01-18 07:30:00', 'free',  'ads',      'android'),  (7,  '2025-01-22 20:15:00', 'paid',  'organic',  'ios'),  (8,  '2025-01-25 16:45:00', 'free',  'referral', 'web'),  (9,  '2025-02-01 09:15:00', 'free',  'ads',      'android'),  (10, '2025-02-05 16:00:00', 'paid',  'referral', 'ios'),  (11, '2025-02-09 10:30:00', 'free',  'organic',  'web'),  (12, '2025-02-12 21:45:00', 'free',  'email',    'android'),  (13, '2025-02-15 07:50:00', 'paid',  'ads',      'web'),  (14, '2025-02-18 13:20:00', 'free',  'organic',  'ios'),  (15, '2025-02-22 19:10:00', 'paid',  'referral', 'web');""")# --- Eventos de enero (usuarios 1..8) ---sql_exec("""INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES  -- Usuario 1: journey COMPLETO  (1,  1, 'session_start',  '2025-01-02 09:12:00', '{"utm":"organic"}'),  (2,  1, 'view_item',      '2025-01-02 09:14:00', '{"sku":"A101"}'),  (3,  1, 'add_to_cart',    '2025-01-02 09:15:00', '{"sku":"A101"}'),  (4,  1, 'begin_checkout', '2025-01-02 09:16:00', '{"cart_value":29.9}'),  (5,  1, 'purchase',       '2025-01-02 09:18:00', '{"order_id":"O-001","value":29.9}'),  -- Retorno semana 1  (6,  1, 'session_start',  '2025-01-09 10:00:00', '{"utm":"email"}'),  (7,  1, 'view_item',      '2025-01-09 10:05:00', '{"sku":"B222"}'),    -- Usuario 2: journey COMPLETO  (8,  2, 'session_start',  '2025-01-03 14:21:00', '{"utm":"ads"}'),  (9,  2, 'view_item',      '2025-01-03 14:22:00', '{"sku":"B222"}'),  (10, 2, 'add_to_cart',    '2025-01-03 14:23:00', '{"sku":"B222"}'),  (11, 2, 'begin_checkout', '2025-01-03 14:24:00', '{"cart_value":58.0}'),  (12, 2, 'purchase',       '2025-01-03 14:27:00', '{"order_id":"O-002","value":58.0}'),  -- Retorno semana 2  (13, 2, 'session_start',  '2025-01-17 08:00:00', '{"utm":"organic"}'),  (14, 2, 'view_item',      '2025-01-17 08:10:00', '{"sku":"C333"}'),  (15, 2, 'purchase',       '2025-01-17 08:20:00', '{"order_id":"O-012","value":45.0}'),    -- Usuario 3: llega hasta checkout (NO compra)  (16, 3, 'session_start',  '2025-01-05 08:06:00', '{"utm":"referral"}'),  (17, 3, 'view_item',      '2025-01-05 08:07:00', '{"sku":"C333"}'),  (18, 3, 'add_to_cart',    '2025-01-05 08:08:00', '{"sku":"C333"}'),  (19, 3, 'begin_checkout', '2025-01-05 08:09:00', '{"cart_value":105.0}'),    -- Usuario 4: solo ve un producto  (20, 4, 'session_start',  '2025-01-10 18:43:00', '{"utm":"organic"}'),  (21, 4, 'view_item',      '2025-01-10 18:46:00', '{"sku":"D444"}'),    -- Usuario 5: agrega al carrito pero no avanza  (22, 5, 'session_start',  '2025-01-12 11:12:00', '{"utm":"email"}'),  (23, 5, 'view_item',      '2025-01-12 11:13:00', '{"sku":"A101"}'),  (24, 5, 'add_to_cart',    '2025-01-12 11:14:00', '{"sku":"A101"}'),  -- Retorno semana 3  (25, 5, 'session_start',  '2025-01-31 15:00:00', '{"utm":"email"}'),    -- Usuario 6: solo session_start  (26, 6, 'session_start',  '2025-01-18 07:32:00', '{"utm":"ads"}'),    -- Usuario 7: journey COMPLETO  (27, 7, 'session_start',  '2025-01-22 20:16:00', '{"utm":"organic"}'),  (28, 7, 'view_item',      '2025-01-22 20:18:00', '{"sku":"E555"}'),  (29, 7, 'add_to_cart',    '2025-01-22 20:19:00', '{"sku":"E555"}'),  (30, 7, 'begin_checkout', '2025-01-22 20:20:00', '{"cart_value":72.5}'),  (31, 7, 'purchase',       '2025-01-22 20:22:00', '{"order_id":"O-007","value":72.5}'),  -- Retorno semana 1 y 3  (32, 7, 'session_start',  '2025-01-29 09:00:00', '{"utm":"organic"}'),  (33, 7, 'view_item',      '2025-01-29 09:05:00', '{"sku":"A101"}'),  (34, 7, 'session_start',  '2025-02-12 11:00:00', '{"utm":"email"}'),    -- Usuario 8: ve productos pero no agrega  (35, 8, 'session_start',  '2025-01-25 16:46:00', '{"utm":"referral"}'),  (36, 8, 'view_item',      '2025-01-25 16:48:00', '{"sku":"F666"}'),  (37, 8, 'view_item',      '2025-01-25 16:50:00', '{"sku":"A101"}');""")# --- Eventos de febrero (usuarios 9..15) ---sql_exec("""INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES  -- Usuario 9: solo ve  (38, 9,  'session_start',  '2025-02-01 09:17:00', '{"utm":"ads"}'),  (39, 9,  'view_item',      '2025-02-01 09:18:00', '{"sku":"E555"}'),    -- Usuario 10: journey COMPLETO  (40, 10, 'session_start',  '2025-02-05 16:01:00', '{"utm":"referral"}'),  (41, 10, 'view_item',      '2025-02-05 16:02:00', '{"sku":"B222"}'),  (42, 10, 'add_to_cart',    '2025-02-05 16:03:00', '{"sku":"B222"}'),  (43, 10, 'begin_checkout', '2025-02-05 16:04:00', '{"cart_value":58.0}'),  (44, 10, 'purchase',       '2025-02-05 16:05:00', '{"order_id":"O-010","value":58.0}'),  -- Retorno semana 1  (45, 10, 'session_start',  '2025-02-12 14:00:00', '{"utm":"organic"}'),  (46, 10, 'view_item',      '2025-02-12 14:10:00', '{"sku":"D444"}'),    -- Usuario 11: agrega al carrito  (47, 11, 'session_start',  '2025-02-09 10:31:00', '{"utm":"organic"}'),  (48, 11, 'view_item',      '2025-02-09 10:32:00', '{"sku":"C333"}'),  (49, 11, 'add_to_cart',    '2025-02-09 10:34:00', '{"sku":"C333"}'),    -- Usuario 12: solo session_start  (50, 12, 'session_start',  '2025-02-12 21:46:00', '{"utm":"email"}'),    -- Usuario 13: journey COMPLETO  (51, 13, 'session_start',  '2025-02-15 07:51:00', '{"utm":"ads"}'),  (52, 13, 'view_item',      '2025-02-15 07:52:00', '{"sku":"A101"}'),  (53, 13, 'add_to_cart',    '2025-02-15 07:53:00', '{"sku":"A101"}'),  (54, 13, 'begin_checkout', '2025-02-15 07:54:00', '{"cart_value":29.9}'),  (55, 13, 'purchase',       '2025-02-15 07:56:00', '{"order_id":"O-013","value":29.9}'),  -- Retorno semana 1  (56, 13, 'session_start',  '2025-02-22 10:00:00', '{"utm":"ads"}'),  (57, 13, 'view_item',      '2025-02-22 10:05:00', '{"sku":"B222"}'),    -- Usuario 14: llega a checkout  (58, 14, 'session_start',  '2025-02-18 13:21:00', '{"utm":"organic"}'),  (59, 14, 'view_item',      '2025-02-18 13:23:00', '{"sku":"D444"}'),  (60, 14, 'add_to_cart',    '2025-02-18 13:25:00', '{"sku":"D444"}'),  (61, 14, 'begin_checkout', '2025-02-18 13:26:00', '{"cart_value":42.0}'),    -- Usuario 15: journey COMPLETO  (62, 15, 'session_start',  '2025-02-22 19:11:00', '{"utm":"referral"}'),  (63, 15, 'view_item',      '2025-02-22 19:13:00', '{"sku":"F666"}'),  (64, 15, 'add_to_cart',    '2025-02-22 19:14:00', '{"sku":"F666"}'),  (65, 15, 'begin_checkout', '2025-02-22 19:15:00', '{"cart_value":89.0}'),  (66, 15, 'purchase',       '2025-02-22 19:17:00', '{"order_id":"O-015","value":89.0}');""")print("✅ Datos insertados: 15 usuarios y 66 eventos.")

In [ ]:
# Verificación rápidaprint("=== Usuarios ===")display(sql("SELECT * FROM users ORDER BY user_id"))print("\n=== Primeros 10 eventos ===")display(sql("SELECT * FROM events ORDER BY event_id LIMIT 10"))

---# 1) ¿Qué es el Data Journey? (20 min)## 1.1 Definición y contextoEl **Data Journey** (o **User Journey**) es la representación del recorrido completo que realiza un usuario a través de un producto o servicio digital, **visto a través de los datos que genera**.Piensa en cada clic, cada página vista, cada compra como una "huella digital" que el usuario deja. Cuando conectamos esas huellas en orden cronológico, obtenemos una historia: el journey.### ¿Por qué es importante?El Data Journey conecta **métricas de producto** con **impacto de negocio**:- **Conversión:** ¿Qué porcentaje de usuarios completa el objetivo? (ej. compra, registro)- **Drop-off:** ¿En qué etapa perdemos más usuarios?- **Tiempo entre etapas:** ¿Cuánto tarda un usuario en decidirse?- **LTV (Lifetime Value):** ¿Cuánto vale un usuario que completa todo el journey?- **Retención:** ¿Vuelve el usuario después de su primera interacción?### AnalogíaImagina un supermercado físico: un cliente entra (session_start), recorre pasillos (view_item), pone productos en el carrito (add_to_cart), va a caja (begin_checkout) y paga (purchase). Si colocas cámaras en cada punto, puedes analizar dónde se van los clientes. **El Data Journey es la versión digital de esas cámaras.**

## 1.2 Las etapas de un Data JourneyUn journey típico de e-commerce tiene estas etapas:```┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌────────────────┐    ┌──────────────┐│ SESSION      │───▶│ VIEW         │───▶│ ADD TO       │───▶│ BEGIN          │───▶│ PURCHASE     ││ START        │    │ ITEM         │    │ CART         │    │ CHECKOUT       │    │              ││              │    │              │    │              │    │                │    │              ││ El usuario   │    │ Explora un   │    │ Muestra      │    │ Inicia el      │    │ Completa la  ││ llega al     │    │ producto     │    │ intención    │    │ proceso de     │    │ transacción  ││ sitio/app    │    │ específico   │    │ de compra    │    │ pago           │    │              │└──────────────┘    └──────────────┘    └──────────────┘    └────────────────┘    └──────────────┘     100%               ~80%                ~50%                 ~30%                  ~20%```Cada flecha representa una **transición** donde podemos perder usuarios. La diferencia entre una etapa y la siguiente es el **drop-off**.### Otros ejemplos de journeys| Industria | Etapa 1 | Etapa 2 | Etapa 3 | Etapa 4 | Etapa 5 ||-----------|---------|---------|---------|---------|---------|| **E-commerce** | session_start | view_item | add_to_cart | begin_checkout | purchase || **Educación** | landing_page | registro | primera_leccion | completar_modulo | certificacion || **Servicios públicos** | portal_ingreso | solicitar_turno | adjuntar_docs | pago | confirmacion || **SaaS** | signup | onboarding | first_action | upgrade | renewal |

## 1.3 Los campos clave para reconstruir un journeyPara reconstruir cualquier journey desde una base de datos necesitas **cuatro campos mínimos**:| Campo | Propósito | Ejemplo ||-------|-----------|---------|| `user_id` | Agrupar eventos por usuario | 1, 2, 3... || `event_ts` | Ordenar cronológicamente | '2025-01-02 09:12:00' || `event_name` | Identificar la etapa/acción | 'session_start', 'purchase' || `props` | Contexto adicional (JSON) | '{"sku":"A101","value":29.9}' |Veamos nuestros datos:

In [ ]:
# Explorar los tipos de eventos disponibles en nuestra basesql("""SELECT     event_name,    COUNT(*) AS total_eventos,    COUNT(DISTINCT user_id) AS usuarios_unicosFROM eventsGROUP BY event_nameORDER BY MIN(event_ts)""")

### 🏋️ Ejercicio 1.1 — Explorar el journey de un usuario específico| | ||---|---|| **Descripción** | Consulta todos los eventos del usuario 1 ordenados cronológicamente para ver su journey completo. || **Objetivo** | Practicar SELECT básico con WHERE y ORDER BY. Visualizar cómo se ve un journey real en datos. || **Pista** | Usa `WHERE user_id = 1` y `ORDER BY event_ts`. || **Explicación** | Cada fila es un paso del usuario en su recorrido. Observa el orden de los eventos y el tiempo entre cada uno. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 1.1)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 1.1)</summary>```pythonsql("""SELECT     event_id,    user_id,    event_name,    event_ts,    propsFROM eventsWHERE user_id = 1ORDER BY event_ts""")```**Interpretación:** El usuario 1 completó el journey completo (session → view → cart → checkout → purchase) el 2 de enero, y regresó una semana después para ver otro producto. ¡Es un usuario retenido!</details>

## 1.4 Validación de calidad de datosAntes de construir cualquier análisis de journey, **siempre** debes validar tus datos. Los problemas más comunes son:1. **Duplicados:** ¿Hay eventos idénticos registrados múltiples veces?2. **Etapas faltantes:** ¿Hay usuarios con `purchase` pero sin `session_start`?3. **Timestamps inválidos:** ¿Hay fechas en el futuro o fuera de rango?### 🏋️ Ejercicio 1.2 — Detectar duplicados| | ||---|---|| **Descripción** | Busca si existen eventos duplicados (mismo user_id, event_name y event_ts). || **Objetivo** | Aprender a usar GROUP BY con HAVING para detectar anomalías en datos. || **Pista** | Agrupa por `user_id, event_name, event_ts` y filtra con `HAVING COUNT(*) > 1`. || **Explicación** | Los duplicados pueden inflar métricas artificialmente. Si un "purchase" se registró dos veces, el funnel mostrará datos incorrectos. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 1.2)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 1.2)</summary>```pythonsql("""SELECT     user_id,     event_name,     event_ts,     COUNT(*) AS nFROM eventsGROUP BY user_id, event_name, event_tsHAVING COUNT(*) > 1ORDER BY n DESC""")```**Interpretación:** Si el resultado está vacío, ¡excelente! No hay duplicados en nuestro dataset. En datos reales, esto casi nunca pasa — siempre debes verificar.</details>

### 🏋️ Ejercicio 1.3 — Detectar journeys "rotos"| | ||---|---|| **Descripción** | Identifica usuarios que tienen `add_to_cart` pero NO tienen `begin_checkout` ni `purchase`. Son usuarios que abandonaron el carrito. || **Objetivo** | Usar subconsultas o JOINs para comparar presencia/ausencia de eventos por usuario. || **Pista** | Usa `NOT IN` o `LEFT JOIN ... WHERE ... IS NULL` para encontrar usuarios que están en un grupo pero no en otro. || **Explicación** | Estos usuarios representan oportunidades de negocio: se puede enviar un email de remarketing o analizar qué los detuvo. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 1.3)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 1.3)</summary>```pythonsql("""SELECT DISTINCT e_cart.user_idFROM events e_cartWHERE e_cart.event_name = 'add_to_cart'  AND e_cart.user_id NOT IN (      SELECT DISTINCT user_id       FROM events       WHERE event_name IN ('begin_checkout', 'purchase')  )ORDER BY e_cart.user_id""")```**Interpretación:** Estos usuarios mostraron intención de compra (agregaron al carrito) pero nunca avanzaron al checkout. Son candidatos ideales para campañas de recuperación de carrito abandonado.</details>

---# 2) CTEs — Common Table Expressions (20 min)## 2.1 ¿Qué es una CTE?Una **CTE (Common Table Expression)** es una consulta temporal con nombre que puedes usar dentro de una sentencia SQL más grande. Se define con la cláusula `WITH`.### Sintaxis básica```sqlWITH nombre_cte AS (    SELECT ...    FROM ...    WHERE ...)SELECT * FROM nombre_cte;```### ¿Por qué usar CTEs?1. **Legibilidad:** Divides una consulta compleja en pasos lógicos con nombres descriptivos.2. **Mantenibilidad:** Es más fácil modificar un paso sin romper los demás.3. **Reutilización:** Puedes referenciar la misma CTE múltiples veces en la consulta principal.4. **Debugging:** Puedes probar cada CTE por separado.### CTE vs Subconsulta| Característica | CTE (`WITH ... AS`) | Subconsulta (`SELECT ... FROM (SELECT ...)`) ||---|---|---|| Legibilidad | ✅ Muy legible | ❌ Se anida y complica || Reutilización | ✅ Se puede referenciar varias veces | ❌ Hay que repetir || Debugging | ✅ Fácil aislar cada paso | ❌ Difícil || Rendimiento | Similar | Similar |

## 2.2 Tu primera CTE — Nivel básicoEmpecemos con algo simple: crear una CTE que seleccione solo los usuarios del plan `free`.### 🏋️ Ejercicio 2.1 — CTE básica con filtro| | ||---|---|| **Descripción** | Crea una CTE llamada `usuarios_free` que filtre solo los usuarios con plan 'free', y luego cuenta cuántos son. || **Objetivo** | Entender la sintaxis básica de `WITH ... AS (...)`. || **Pista** | Define `WITH usuarios_free AS (SELECT ... WHERE plan = 'free')` y luego haz `SELECT COUNT(*) FROM usuarios_free`. || **Explicación** | Aunque este ejemplo es simple, la estructura es exactamente la misma que usarás para consultas complejas. El patrón siempre es: definir → usar. |

In [ ]:
# Ejemplo resuelto para que veas la estructura:sql("""WITH usuarios_free AS (    SELECT user_id, signup_ts, channel, device    FROM users    WHERE plan = 'free')SELECT COUNT(*) AS total_freeFROM usuarios_free""")

In [ ]:
# ══════════════════════════════════════════# 👇 Ahora tú: crea una CTE de usuarios 'paid'#    y muestra su user_id, signup_ts y device#    (Ejercicio 2.1)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 2.1)</summary>```pythonsql("""WITH usuarios_paid AS (    SELECT user_id, signup_ts, device    FROM users    WHERE plan = 'paid')SELECT * FROM usuarios_paidORDER BY signup_ts""")```</details>

## 2.3 Múltiples CTEs encadenadas — Nivel intermedioPuedes definir **varias CTEs** separadas por comas. Cada CTE puede referenciar a las anteriores.```sqlWITH cte_primera AS (    SELECT ...),cte_segunda AS (    SELECT ...    FROM cte_primera   -- ← puede usar la CTE anterior),cte_tercera AS (    SELECT ...    FROM cte_segunda)SELECT * FROM cte_tercera;```### 🏋️ Ejercicio 2.2 — Dos CTEs encadenadas| | ||---|---|| **Descripción** | Crea dos CTEs: (1) `eventos_enero` que filtre eventos de enero 2025, y (2) `resumen_enero` que cuente eventos por tipo a partir de la primera CTE. || **Objetivo** | Practicar el encadenamiento de CTEs donde la segunda usa la primera. || **Pista** | En la primera CTE filtra con `event_ts >= '2025-01-01' AND event_ts < '2025-02-01'`. En la segunda, haz `GROUP BY event_name` sobre `eventos_enero`. || **Explicación** | Este patrón es fundamental: primero filtras/preparas datos, luego los agregas o transformas. Es como una línea de montaje donde cada paso refina el anterior. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 2.2)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 2.2)</summary>```pythonsql("""WITH eventos_enero AS (    SELECT *    FROM events    WHERE event_ts >= '2025-01-01'       AND event_ts < '2025-02-01'),resumen_enero AS (    SELECT         event_name,        COUNT(*) AS total_eventos,        COUNT(DISTINCT user_id) AS usuarios_unicos    FROM eventos_enero    GROUP BY event_name)SELECT * FROM resumen_eneroORDER BY total_eventos DESC""")```**Interpretación:** Puedes ver cuántos eventos de cada tipo ocurrieron en enero y cuántos usuarios únicos participaron en cada etapa. Esto ya te da una idea del funnel.</details>

## 2.4 CTEs con JOINs — Nivel intermedio-avanzadoLas CTEs brillan cuando necesitas combinar datos de diferentes fuentes o perspectivas.### 🏋️ Ejercicio 2.3 — CTE con JOIN a tabla de usuarios| | ||---|---|| **Descripción** | Crea una CTE que contenga los eventos de `purchase` junto con el `plan` y `channel` del usuario (de la tabla `users`). Luego, cuenta cuántas compras hubo por canal de adquisición. || **Objetivo** | Combinar CTEs con JOINs para enriquecer datos de eventos con atributos de usuario. || **Pista** | En la CTE, haz `JOIN users ON events.user_id = users.user_id`. En la consulta final, agrupa por `channel`. || **Explicación** | En la vida real, los datos de eventos y los atributos de usuario suelen vivir en tablas separadas. Las CTEs te permiten unirlos de forma ordenada antes de hacer el análisis. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 2.3)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 2.3)</summary>```pythonsql("""WITH compras_enriquecidas AS (    SELECT         e.event_id,        e.user_id,        e.event_ts,        e.props,        u.plan,        u.channel,        u.device    FROM events e    JOIN users u ON e.user_id = u.user_id    WHERE e.event_name = 'purchase')SELECT     channel,    COUNT(*) AS total_compras,    COUNT(DISTINCT user_id) AS compradores_unicosFROM compras_enriquecidasGROUP BY channelORDER BY total_compras DESC""")```**Interpretación:** Ahora puedes ver qué canal de adquisición genera más compras. Esto es clave para decidir dónde invertir presupuesto de marketing.</details>

## 2.5 CTEs avanzadas — Múltiples CTEs independientesA veces necesitas CTEs que **no dependen** entre sí, sino que se combinan al final. Este es el patrón clave para los funnels.### 🏋️ Ejercicio 2.4 — Tres CTEs independientes combinadas| | ||---|---|| **Descripción** | Crea tres CTEs independientes: (1) usuarios que hicieron `session_start`, (2) usuarios que hicieron `view_item`, (3) usuarios que hicieron `purchase`. Luego, muestra el conteo de cada grupo en una sola fila. || **Objetivo** | Dominar el patrón de "una CTE por etapa" que usaremos extensamente en funnels. || **Pista** | Cada CTE selecciona `DISTINCT user_id` filtrado por `event_name`. La consulta final usa subconsultas `(SELECT COUNT(*) FROM cte_x)`. || **Explicación** | Este patrón es la **base de todo funnel analysis**. Cada CTE aísla una etapa del journey, y al final las combinamos para ver la progresión. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 2.4)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 2.4)</summary>```pythonsql("""WITH cte_session AS (    SELECT DISTINCT user_id FROM events WHERE event_name = 'session_start'),cte_view AS (    SELECT DISTINCT user_id FROM events WHERE event_name = 'view_item'),cte_purchase AS (    SELECT DISTINCT user_id FROM events WHERE event_name = 'purchase')SELECT    (SELECT COUNT(*) FROM cte_session)  AS users_session,    (SELECT COUNT(*) FROM cte_view)     AS users_view,    (SELECT COUNT(*) FROM cte_purchase) AS users_purchase""")```**Interpretación:** De los 15 usuarios que iniciaron sesión, X vieron un producto y Y compraron. La diferencia entre cada número es el drop-off. ¡Esto ya es un mini-funnel!</details>

---# 3) Funnel Analysis con SQL (20 min)## 3.1 ¿Qué es un funnel de conversión?Un **funnel** (embudo) de conversión toma el journey y lo cuantifica: ¿cuántos usuarios pasan de cada etapa a la siguiente?```  session_start   ████████████████████████████  15 usuarios (100%)  view_item       ████████████████████████       13 usuarios  (87%)  add_to_cart     ████████████████               9 usuarios   (60%)  begin_checkout  ████████████                   7 usuarios   (47%)  purchase        ████████                       6 usuarios   (40%)```Las métricas clave del funnel son:- **Drop-off:** Usuarios perdidos entre etapa N y etapa N+1.- **Tasa de conversión por etapa:** % que avanza a la siguiente etapa.- **Tasa de conversión total:** % que completa el journey desde el inicio.

## 3.2 Construir un funnel completo con CTEsAhora combinamos todo lo aprendido. El patrón es:1. **CTE base:** filtrar la ventana temporal.2. **Una CTE por etapa:** `SELECT DISTINCT user_id` por cada `event_name`.3. **Consulta final:** contar usuarios por etapa y calcular métricas.

In [ ]:
# Funnel completo con conteos por etapasql("""WITH base AS (    SELECT *    FROM events    WHERE event_ts >= '2025-01-01'      AND event_ts <  '2025-03-01'),cte_session AS (    SELECT DISTINCT user_id FROM base WHERE event_name = 'session_start'),cte_view AS (    SELECT DISTINCT user_id FROM base WHERE event_name = 'view_item'),cte_cart AS (    SELECT DISTINCT user_id FROM base WHERE event_name = 'add_to_cart'),cte_checkout AS (    SELECT DISTINCT user_id FROM base WHERE event_name = 'begin_checkout'),cte_purchase AS (    SELECT DISTINCT user_id FROM base WHERE event_name = 'purchase')SELECT    (SELECT COUNT(*) FROM cte_session)  AS session_users,    (SELECT COUNT(*) FROM cte_view)     AS view_users,    (SELECT COUNT(*) FROM cte_cart)     AS cart_users,    (SELECT COUNT(*) FROM cte_checkout) AS checkout_users,    (SELECT COUNT(*) FROM cte_purchase) AS purchase_users""")

## 3.3 Funnel con drop-off y tasas de conversión### 🏋️ Ejercicio 3.1 — Funnel con métricas completas| | ||---|---|| **Descripción** | Extiende el funnel anterior para calcular: (a) drop-off entre cada etapa, (b) tasa de conversión de cada etapa respecto a la anterior, y (c) tasa de conversión total (session → purchase). || **Objetivo** | Construir un funnel analítico completo que permita identificar cuellos de botella. || **Pista** | Usa una CTE `counts` que reúna los conteos, y luego calcula `n_session - n_view AS drop_session_to_view` y `ROUND(100.0 * n_view / NULLIF(n_session, 0), 2)` para los porcentajes. || **Explicación** | El drop-off absoluto te dice CUÁNTOS usuarios pierdes; el porcentaje te dice QUÉ TAN grave es la pérdida relativa. Ambos son necesarios para priorizar mejoras. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 3.1)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 3.1)</summary>```pythonsql("""WITH base AS (    SELECT * FROM events    WHERE event_ts >= '2025-01-01' AND event_ts < '2025-03-01'),s  AS (SELECT DISTINCT user_id FROM base WHERE event_name = 'session_start'),v  AS (SELECT DISTINCT user_id FROM base WHERE event_name = 'view_item'),c  AS (SELECT DISTINCT user_id FROM base WHERE event_name = 'add_to_cart'),ch AS (SELECT DISTINCT user_id FROM base WHERE event_name = 'begin_checkout'),p  AS (SELECT DISTINCT user_id FROM base WHERE event_name = 'purchase'),counts AS (    SELECT        (SELECT COUNT(*) FROM s)  AS n_session,        (SELECT COUNT(*) FROM v)  AS n_view,        (SELECT COUNT(*) FROM c)  AS n_cart,        (SELECT COUNT(*) FROM ch) AS n_checkout,        (SELECT COUNT(*) FROM p)  AS n_purchase)SELECT    n_session,    n_view,    n_cart,    n_checkout,    n_purchase,    -- Drop-off absoluto    (n_session - n_view)      AS drop_session_view,    (n_view    - n_cart)      AS drop_view_cart,    (n_cart    - n_checkout)  AS drop_cart_checkout,    (n_checkout - n_purchase) AS drop_checkout_purchase,    -- Tasas de conversión entre etapas    ROUND(100.0 * n_view     / NULLIF(n_session, 0),  2) AS cr_session_to_view,    ROUND(100.0 * n_cart     / NULLIF(n_view, 0),     2) AS cr_view_to_cart,    ROUND(100.0 * n_checkout / NULLIF(n_cart, 0),     2) AS cr_cart_to_checkout,    ROUND(100.0 * n_purchase / NULLIF(n_checkout, 0), 2) AS cr_checkout_to_purchase,    -- Conversión total    ROUND(100.0 * n_purchase / NULLIF(n_session, 0),  2) AS cr_total_pctFROM counts""")```</details>

## 3.4 Funnel segmentado por atributo### 🏋️ Ejercicio 3.2 — Funnel por plan (free vs paid)| | ||---|---|| **Descripción** | Construye un funnel que muestre la conversión por etapa separada por plan (`free` vs `paid`). || **Objetivo** | Aprender a segmentar un funnel por un atributo de usuario para comparar comportamientos entre grupos. || **Pista** | En la CTE `base`, haz un JOIN con `users` para obtener el `plan`. Luego, en cada CTE de etapa, incluye `plan` en el `SELECT DISTINCT`. Al final, agrupa por `plan`. || **Explicación** | Segmentar es clave: quizás los usuarios `paid` convierten mucho más que los `free`, lo cual justifica invertir en upselling. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 3.2)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 3.2)</summary>```pythonsql("""WITH base AS (    SELECT e.*, u.plan    FROM events e    JOIN users u ON e.user_id = u.user_id    WHERE e.event_ts >= '2025-01-01' AND e.event_ts < '2025-03-01')SELECT    plan,    COUNT(DISTINCT CASE WHEN event_name = 'session_start'  THEN user_id END) AS n_session,    COUNT(DISTINCT CASE WHEN event_name = 'view_item'      THEN user_id END) AS n_view,    COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart'    THEN user_id END) AS n_cart,    COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN user_id END) AS n_checkout,    COUNT(DISTINCT CASE WHEN event_name = 'purchase'       THEN user_id END) AS n_purchase,    ROUND(100.0 * COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END)         / NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'session_start' THEN user_id END), 0), 2    ) AS cr_total_pctFROM baseGROUP BY planORDER BY plan""")```**Interpretación:** Compara la tasa de conversión total entre `free` y `paid`. ¿Quién convierte más? ¿Tiene sentido ofrecer incentivos a los usuarios free para que compren?</details>

## 3.5 Interpretación de resultados del funnelUna vez que tienes el funnel construido, las preguntas de negocio son:1. **¿Cuál es el cuello de botella?** — La etapa con mayor drop-off porcentual.2. **¿Por qué se van?** — Hipótesis: fricción en checkout móvil, precios altos, falta de medios de pago, lentitud del sitio.3. **¿Qué hacer?** — Proponer: test A/B en esa etapa, UX research con usuarios que abandonaron, remarketing por email.> **Regla de oro:** Mejorar el cuello de botella genera el mayor impacto con el menor esfuerzo.

---# 4) Análisis de Retención con Cohorts (20 min)## 4.1 ¿Qué es un cohort?Un **cohort** es un grupo de usuarios que comparten una característica en un momento determinado. Lo más común es agrupar por **fecha de registro** (signup).- **Cohort de enero 2025:** Todos los usuarios que se registraron en enero 2025.- **Cohort de febrero 2025:** Todos los usuarios que se registraron en febrero 2025.### ¿Qué medimos?La **retención**: ¿Cuántos usuarios de cada cohort vuelven a realizar alguna acción en los periodos siguientes?```             Semana 0    Semana 1    Semana 2    Semana 3Cohort Ene   100%        40%         20%         10%Cohort Feb   100%        50%         30%         15%```Una tabla así nos dice: "De los usuarios de enero, el 40% volvió en la semana 1, pero solo el 10% en la semana 3." Esto es fundamental para entender si nuestro producto retiene usuarios.

## 4.2 Retención semanal por cohort (tabla larga)La consulta usa varias CTEs:1. **`u`** — Determinar la cohorte mensual de cada usuario.2. **`grid`** — Generar una secuencia de semanas (0, 1, 2, ..., 8).3. **`retention`** — Para cada combinación (cohorte, semana), contar usuarios retenidos.4. **Consulta final** — Calcular el porcentaje.

In [ ]:
# Retención semanal por cohort (tabla larga)sql("""WITH-- 1) Cohorte mensual = primer día del mes del signupu AS (    SELECT        user_id,        DATE_TRUNC('month', signup_ts)::DATE AS cohort_month    FROM users),-- 2) Grid de semanas 0..8grid AS (    SELECT UNNEST(RANGE(0, 9)) AS week_offset),-- 3) Retención: ¿el usuario tuvo al menos 1 evento en la semana w?retention AS (    SELECT        u.cohort_month,        g.week_offset,        COUNT(DISTINCT u.user_id) FILTER (            WHERE EXISTS (                SELECT 1                FROM events e                WHERE e.user_id = u.user_id                  AND e.event_ts::DATE >= u.cohort_month + (g.week_offset * 7)                  AND e.event_ts::DATE <  u.cohort_month + ((g.week_offset + 1) * 7)            )        ) AS retained_users,        COUNT(DISTINCT u.user_id) AS cohort_users    FROM u    CROSS JOIN grid g    GROUP BY u.cohort_month, g.week_offset)-- 4) PorcentajeSELECT    cohort_month,    week_offset,    retained_users,    cohort_users,    ROUND(100.0 * retained_users / NULLIF(cohort_users, 0), 2) AS retention_pctFROM retentionORDER BY cohort_month, week_offset""")

## 4.3 Retención pivoteada (tabla ancha — ideal para heatmap)### 🏋️ Ejercicio 4.1 — Pivotar la tabla de retención| | ||---|---|| **Descripción** | Transforma la tabla larga de retención en una tabla ancha donde cada fila es un cohort y cada columna es una semana (w0, w1, ..., w8). || **Objetivo** | Aprender a hacer un pivote manual en SQL usando `MAX(CASE WHEN ... END)`. Este formato es perfecto para exportar a Google Sheets y crear un heatmap. || **Pista** | Usa `MAX(CASE WHEN week_offset = 0 THEN retention_pct END) AS w0` para cada semana. Agrupa por `cohort_month`. || **Explicación** | Las tablas anchas son más fáciles de visualizar: cada celda muestra el % de retención para una cohorte en una semana específica. Con formato condicional (colores), se convierte en un heatmap. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 4.1)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 4.1)</summary>```pythonsql("""WITHu AS (    SELECT user_id, DATE_TRUNC('month', signup_ts)::DATE AS cohort_month    FROM users),grid AS (    SELECT UNNEST(RANGE(0, 9)) AS week_offset),retention AS (    SELECT        u.cohort_month,        g.week_offset,        COUNT(DISTINCT u.user_id) FILTER (            WHERE EXISTS (                SELECT 1 FROM events e                WHERE e.user_id = u.user_id                  AND e.event_ts::DATE >= u.cohort_month + (g.week_offset * 7)                  AND e.event_ts::DATE <  u.cohort_month + ((g.week_offset + 1) * 7)            )        ) AS retained_users,        COUNT(DISTINCT u.user_id) AS cohort_users    FROM u CROSS JOIN grid g    GROUP BY u.cohort_month, g.week_offset),base AS (    SELECT        cohort_month,        week_offset,        ROUND(100.0 * retained_users / NULLIF(cohort_users, 0), 2) AS retention_pct    FROM retention)SELECT    cohort_month,    MAX(CASE WHEN week_offset = 0 THEN retention_pct END) AS w0,    MAX(CASE WHEN week_offset = 1 THEN retention_pct END) AS w1,    MAX(CASE WHEN week_offset = 2 THEN retention_pct END) AS w2,    MAX(CASE WHEN week_offset = 3 THEN retention_pct END) AS w3,    MAX(CASE WHEN week_offset = 4 THEN retention_pct END) AS w4,    MAX(CASE WHEN week_offset = 5 THEN retention_pct END) AS w5,    MAX(CASE WHEN week_offset = 6 THEN retention_pct END) AS w6,    MAX(CASE WHEN week_offset = 7 THEN retention_pct END) AS w7,    MAX(CASE WHEN week_offset = 8 THEN retention_pct END) AS w8FROM baseGROUP BY cohort_monthORDER BY cohort_month""")```**Tip:** Exporta este resultado a CSV y ábrelo en Google Sheets. Selecciona las columnas w0..w8, aplica Formato condicional → Escala de color, y tendrás un heatmap visual de retención.</details>

## 4.4 Retención segmentada por plan (free vs paid)### 🏋️ Ejercicio 4.2 — Retención por plan| | ||---|---|| **Descripción** | Modifica la consulta de retención para incluir el `plan` del usuario como dimensión adicional. El resultado debe mostrar retención por (cohort_month, plan, week_offset). || **Objetivo** | Segmentar la retención por un atributo de negocio para comparar comportamiento entre grupos. || **Pista** | Agrega `plan` al SELECT de la CTE `u`, inclúyelo en el GROUP BY de `retention`, y en la consulta final. || **Explicación** | Si los usuarios `paid` tienen mejor retención, confirma que el modelo de monetización funciona. Si los `free` retienen mejor, quizás el upgrade los ahuyenta. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 4.2)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 4.2)</summary>```pythonsql("""WITHu AS (    SELECT user_id, DATE_TRUNC('month', signup_ts)::DATE AS cohort_month, plan    FROM users),grid AS (    SELECT UNNEST(RANGE(0, 9)) AS week_offset),retention AS (    SELECT        u.cohort_month,        u.plan,        g.week_offset,        COUNT(DISTINCT u.user_id) FILTER (            WHERE EXISTS (                SELECT 1 FROM events e                WHERE e.user_id = u.user_id                  AND e.event_ts::DATE >= u.cohort_month + (g.week_offset * 7)                  AND e.event_ts::DATE <  u.cohort_month + ((g.week_offset + 1) * 7)            )        ) AS retained_users,        COUNT(DISTINCT u.user_id) AS cohort_users    FROM u CROSS JOIN grid g    GROUP BY u.cohort_month, u.plan, g.week_offset)SELECT    cohort_month,    plan,    week_offset,    retained_users,    cohort_users,    ROUND(100.0 * retained_users / NULLIF(cohort_users, 0), 2) AS retention_pctFROM retentionORDER BY cohort_month, plan, week_offset""")```</details>

---# 5) Ejercicios Integradores (10 min)Estos ejercicios combinan todo lo visto. La dificultad es creciente.

### 🏋️ Ejercicio 5.1 — Funnel solo de febrero 2025 (⭐ Fácil)| | ||---|---|| **Descripción** | Construye un funnel de 5 etapas (session → view → cart → checkout → purchase) pero solo para eventos de **febrero 2025**. || **Objetivo** | Reforzar la construcción de funnels filtrando por ventana temporal. || **Pista** | Modifica la CTE `base` para filtrar `event_ts >= '2025-02-01' AND event_ts < '2025-03-01'`. || **Explicación** | Comparar funnels de distintos periodos te permite detectar si hay mejoras o empeoramiento en la conversión mes a mes. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 5.1)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 5.1)</summary>```pythonsql("""WITH base AS (    SELECT * FROM events    WHERE event_ts >= '2025-02-01' AND event_ts < '2025-03-01'),s  AS (SELECT DISTINCT user_id FROM base WHERE event_name = 'session_start'),v  AS (SELECT DISTINCT user_id FROM base WHERE event_name = 'view_item'),c  AS (SELECT DISTINCT user_id FROM base WHERE event_name = 'add_to_cart'),ch AS (SELECT DISTINCT user_id FROM base WHERE event_name = 'begin_checkout'),p  AS (SELECT DISTINCT user_id FROM base WHERE event_name = 'purchase')SELECT    (SELECT COUNT(*) FROM s)  AS session_users,    (SELECT COUNT(*) FROM v)  AS view_users,    (SELECT COUNT(*) FROM c)  AS cart_users,    (SELECT COUNT(*) FROM ch) AS checkout_users,    (SELECT COUNT(*) FROM p)  AS purchase_users,    ROUND(100.0 * (SELECT COUNT(*) FROM p) / NULLIF((SELECT COUNT(*) FROM s), 0), 2) AS cr_total_pct""")```</details>

### 🏋️ Ejercicio 5.2 — Funnel por dispositivo (⭐⭐ Intermedio)| | ||---|---|| **Descripción** | Construye un funnel segmentado por `device` (web, android, ios). Muestra cuántos usuarios llegan a cada etapa por dispositivo y la conversión total. || **Objetivo** | Aplicar el patrón de funnel segmentado con un atributo diferente al plan. || **Pista** | Usa `COUNT(DISTINCT CASE WHEN event_name = '...' THEN user_id END)` agrupado por `device`. || **Explicación** | Si los usuarios de móvil (android/ios) convierten menos que los de web, podría haber un problema de UX en la versión móvil — especialmente en el checkout. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 5.2)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 5.2)</summary>```pythonsql("""WITH base AS (    SELECT e.*, u.device    FROM events e    JOIN users u ON e.user_id = u.user_id    WHERE e.event_ts >= '2025-01-01' AND e.event_ts < '2025-03-01')SELECT    device,    COUNT(DISTINCT CASE WHEN event_name = 'session_start'  THEN user_id END) AS n_session,    COUNT(DISTINCT CASE WHEN event_name = 'view_item'      THEN user_id END) AS n_view,    COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart'    THEN user_id END) AS n_cart,    COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN user_id END) AS n_checkout,    COUNT(DISTINCT CASE WHEN event_name = 'purchase'       THEN user_id END) AS n_purchase,    ROUND(100.0 * COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END)         / NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'session_start' THEN user_id END), 0), 2    ) AS cr_total_pctFROM baseGROUP BY deviceORDER BY n_session DESC""")```</details>

### 🏋️ Ejercicio 5.3 — Tiempo promedio entre etapas (⭐⭐⭐ Avanzado)| | ||---|---|| **Descripción** | Calcula el **tiempo promedio** (en minutos) que tarda un usuario en pasar de `session_start` a `purchase`. Solo incluye usuarios que completaron ambas etapas. || **Objetivo** | Usar CTEs con JOINs y funciones de fecha para medir tiempos entre eventos. || **Pista** | Crea una CTE para el primer `session_start` de cada usuario y otra para su primer `purchase`. Haz JOIN por `user_id` y calcula la diferencia en minutos con `DATE_DIFF('minute', ts_session, ts_purchase)`. || **Explicación** | El tiempo de conversión es una métrica oculta pero poderosa. Si un usuario tarda mucho en comprar, hay fricción. Si es muy rápido, el producto es convincente. |

In [ ]:
# ══════════════════════════════════════════# 👇 Tu código aquí (Ejercicio 5.3)# ══════════════════════════════════════════

<details><summary>💡 Ver solución (Ejercicio 5.3)</summary>```pythonsql("""WITH first_session AS (    SELECT user_id, MIN(event_ts) AS ts_session    FROM events    WHERE event_name = 'session_start'    GROUP BY user_id),first_purchase AS (    SELECT user_id, MIN(event_ts) AS ts_purchase    FROM events    WHERE event_name = 'purchase'    GROUP BY user_id),tiempos AS (    SELECT         s.user_id,        s.ts_session,        p.ts_purchase,        DATE_DIFF('minute', s.ts_session, p.ts_purchase) AS minutos_a_compra    FROM first_session s    JOIN first_purchase p ON s.user_id = p.user_id)SELECT     COUNT(*) AS usuarios_compradores,    ROUND(AVG(minutos_a_compra), 2) AS avg_minutos,    MIN(minutos_a_compra) AS min_minutos,    MAX(minutos_a_compra) AS max_minutosFROM tiempos""")```**Interpretación:** Si el promedio es de pocos minutos, los compradores son decisivos. Si hay gran variación (MIN muy bajo, MAX muy alto), hay segmentos muy diferentes que vale la pena investigar.</details>

---# 📋 Apéndice: Scripts SQL portables (para SQL Workbench / SQLite)Si prefieres ejecutar estas consultas fuera de este notebook, aquí tienes los scripts de creación e inserción adaptados para **SQLite** (compatible con SQL Workbench, DBeaver, sqliteonline.com).> Copia y pega estos scripts en tu herramienta SQL favorita.

In [ ]:
# Script DDL + INSERT para SQLite / SQL Workbench# (Cópialo y ejecútalo en tu herramienta SQL externa)print(r'''-- ============================================-- DDL: Crear tablas (compatible SQLite)-- ============================================CREATE TABLE IF NOT EXISTS users (  user_id     INTEGER PRIMARY KEY AUTOINCREMENT,  signup_ts   TEXT NOT NULL,  plan        TEXT NOT NULL CHECK (plan IN ('free','paid')),  channel     TEXT NOT NULL CHECK (channel IN ('organic','ads','referral','email')),  device      TEXT NOT NULL CHECK (device IN ('web','android','ios')));CREATE TABLE IF NOT EXISTS events (  event_id    INTEGER PRIMARY KEY AUTOINCREMENT,  user_id     INTEGER REFERENCES users(user_id),  event_name  TEXT NOT NULL,  event_ts    TEXT NOT NULL,  props       TEXT NOT NULL DEFAULT '{}');CREATE INDEX IF NOT EXISTS idx_events_user_ts ON events (user_id, event_ts);CREATE INDEX IF NOT EXISTS idx_events_name_ts ON events (event_name, event_ts);-- ============================================-- SEED: Insertar datos de ejemplo-- ============================================INSERT INTO users (signup_ts, plan, channel, device) VALUES  ('2025-01-02 09:10:00','free','organic','web'),  ('2025-01-03 14:20:00','paid','ads','ios'),  ('2025-01-05 08:05:00','free','referral','android'),  ('2025-01-10 18:42:00','free','organic','web'),  ('2025-01-12 11:11:00','paid','email','web'),  ('2025-01-18 07:30:00','free','ads','android'),  ('2025-01-22 20:15:00','paid','organic','ios'),  ('2025-01-25 16:45:00','free','referral','web'),  ('2025-02-01 09:15:00','free','ads','android'),  ('2025-02-05 16:00:00','paid','referral','ios'),  ('2025-02-09 10:30:00','free','organic','web'),  ('2025-02-12 21:45:00','free','email','android'),  ('2025-02-15 07:50:00','paid','ads','web'),  ('2025-02-18 13:20:00','free','organic','ios'),  ('2025-02-22 19:10:00','paid','referral','web');INSERT INTO events (user_id, event_name, event_ts, props) VALUES  (1,'session_start','2025-01-02 09:12:00','{"utm":"organic"}'),  (1,'view_item','2025-01-02 09:14:00','{"sku":"A101"}'),  (1,'add_to_cart','2025-01-02 09:15:00','{"sku":"A101"}'),  (1,'begin_checkout','2025-01-02 09:16:00','{"cart_value":29.9}'),  (1,'purchase','2025-01-02 09:18:00','{"order_id":"O-001","value":29.9}'),  (1,'session_start','2025-01-09 10:00:00','{"utm":"email"}'),  (1,'view_item','2025-01-09 10:05:00','{"sku":"B222"}'),  (2,'session_start','2025-01-03 14:21:00','{"utm":"ads"}'),  (2,'view_item','2025-01-03 14:22:00','{"sku":"B222"}'),  (2,'add_to_cart','2025-01-03 14:23:00','{"sku":"B222"}'),  (2,'begin_checkout','2025-01-03 14:24:00','{"cart_value":58.0}'),  (2,'purchase','2025-01-03 14:27:00','{"order_id":"O-002","value":58.0}'),  (2,'session_start','2025-01-17 08:00:00','{"utm":"organic"}'),  (2,'view_item','2025-01-17 08:10:00','{"sku":"C333"}'),  (2,'purchase','2025-01-17 08:20:00','{"order_id":"O-012","value":45.0}'),  (3,'session_start','2025-01-05 08:06:00','{"utm":"referral"}'),  (3,'view_item','2025-01-05 08:07:00','{"sku":"C333"}'),  (3,'add_to_cart','2025-01-05 08:08:00','{"sku":"C333"}'),  (3,'begin_checkout','2025-01-05 08:09:00','{"cart_value":105.0}'),  (4,'session_start','2025-01-10 18:43:00','{"utm":"organic"}'),  (4,'view_item','2025-01-10 18:46:00','{"sku":"D444"}'),  (5,'session_start','2025-01-12 11:12:00','{"utm":"email"}'),  (5,'view_item','2025-01-12 11:13:00','{"sku":"A101"}'),  (5,'add_to_cart','2025-01-12 11:14:00','{"sku":"A101"}'),  (5,'session_start','2025-01-31 15:00:00','{"utm":"email"}'),  (6,'session_start','2025-01-18 07:32:00','{"utm":"ads"}'),  (7,'session_start','2025-01-22 20:16:00','{"utm":"organic"}'),  (7,'view_item','2025-01-22 20:18:00','{"sku":"E555"}'),  (7,'add_to_cart','2025-01-22 20:19:00','{"sku":"E555"}'),  (7,'begin_checkout','2025-01-22 20:20:00','{"cart_value":72.5}'),  (7,'purchase','2025-01-22 20:22:00','{"order_id":"O-007","value":72.5}'),  (7,'session_start','2025-01-29 09:00:00','{"utm":"organic"}'),  (7,'view_item','2025-01-29 09:05:00','{"sku":"A101"}'),  (7,'session_start','2025-02-12 11:00:00','{"utm":"email"}'),  (8,'session_start','2025-01-25 16:46:00','{"utm":"referral"}'),  (8,'view_item','2025-01-25 16:48:00','{"sku":"F666"}'),  (8,'view_item','2025-01-25 16:50:00','{"sku":"A101"}'),  (9,'session_start','2025-02-01 09:17:00','{"utm":"ads"}'),  (9,'view_item','2025-02-01 09:18:00','{"sku":"E555"}'),  (10,'session_start','2025-02-05 16:01:00','{"utm":"referral"}'),  (10,'view_item','2025-02-05 16:02:00','{"sku":"B222"}'),  (10,'add_to_cart','2025-02-05 16:03:00','{"sku":"B222"}'),  (10,'begin_checkout','2025-02-05 16:04:00','{"cart_value":58.0}'),  (10,'purchase','2025-02-05 16:05:00','{"order_id":"O-010","value":58.0}'),  (10,'session_start','2025-02-12 14:00:00','{"utm":"organic"}'),  (10,'view_item','2025-02-12 14:10:00','{"sku":"D444"}'),  (11,'session_start','2025-02-09 10:31:00','{"utm":"organic"}'),  (11,'view_item','2025-02-09 10:32:00','{"sku":"C333"}'),  (11,'add_to_cart','2025-02-09 10:34:00','{"sku":"C333"}'),  (12,'session_start','2025-02-12 21:46:00','{"utm":"email"}'),  (13,'session_start','2025-02-15 07:51:00','{"utm":"ads"}'),  (13,'view_item','2025-02-15 07:52:00','{"sku":"A101"}'),  (13,'add_to_cart','2025-02-15 07:53:00','{"sku":"A101"}'),  (13,'begin_checkout','2025-02-15 07:54:00','{"cart_value":29.9}'),  (13,'purchase','2025-02-15 07:56:00','{"order_id":"O-013","value":29.9}'),  (13,'session_start','2025-02-22 10:00:00','{"utm":"ads"}'),  (13,'view_item','2025-02-22 10:05:00','{"sku":"B222"}'),  (14,'session_start','2025-02-18 13:21:00','{"utm":"organic"}'),  (14,'view_item','2025-02-18 13:23:00','{"sku":"D444"}'),  (14,'add_to_cart','2025-02-18 13:25:00','{"sku":"D444"}'),  (14,'begin_checkout','2025-02-18 13:26:00','{"cart_value":42.0}'),  (15,'session_start','2025-02-22 19:11:00','{"utm":"referral"}'),  (15,'view_item','2025-02-22 19:13:00','{"sku":"F666"}'),  (15,'add_to_cart','2025-02-22 19:14:00','{"sku":"F666"}'),  (15,'begin_checkout','2025-02-22 19:15:00','{"cart_value":89.0}'),  (15,'purchase','2025-02-22 19:17:00','{"order_id":"O-015","value":89.0}');''')

---## Cierre (5 min)### Resumen de lo aprendido| Tema | Concepto clave ||------|---------------|| **Data Journey** | Recorrido del usuario en datos: cada evento es una huella que conectamos cronológicamente || **Campos clave** | `user_id`, `event_ts`, `event_name`, `props` — la base de cualquier análisis de journey || **Validación** | Siempre verificar duplicados, faltantes y timestamps antes de analizar || **CTEs** | `WITH nombre AS (...)` — dividen consultas complejas en pasos legibles y mantenibles || **Funnels** | Cuantificar cuántos usuarios pasan de cada etapa a la siguiente (drop-off + conversión) || **Cohorts** | Agrupar usuarios por fecha de registro y medir retención en periodos posteriores |### Reflexión- ¿Por qué el análisis de funnels es crítico para productos digitales?- ¿Qué ventaja tienen las CTEs para organizar tu código SQL?- ¿Cómo usarías la retención por cohort para evaluar si un cambio en tu producto fue exitoso?### Q&A y próximos pasos

## Siguientes Pasos- **Próxima sesión:** Sprint 4 — Práctica de construcción de Funnels en SQL.- **Tarea:** Visualiza un funnel de un producto que uses (ej. Netflix, Amazon, Spotify) e identifica las posibles etapas del journey.- **Recordatorios:** Repasa la sintaxis de `WITH` (CTE) — es la herramienta más importante que aprenderás en SQL analítico.- **Recurso extra:** Practica en https://sqliteonline.com/ con los scripts del apéndice.